# Phase 3.2：单变量实验：top-k 与分块参数

## 目标

一次只改变一个主要变量，同时记录质量和性能。我们会比较不同 `top_k`，再比较不同 Chunk 配置，理解“更快”和“更好”可能互相冲突。

**本课交付：** `data/processed/phase3_single_variable_experiments.json`。

## Evidence Quest 任务卡：Phase 3.2：参数改装实验室

**你的身份：** 实验车辆调参师  
**案件背景：** 你有一台会漏证据的搜索引擎。现在只能一次改一个旋钮，找出速度、召回和上下文噪声之间更值得采用的配置。

### 本关专业 Goal

用单变量实验比较 top-k、chunk_size 和 overlap，形成有边界的工程结论。

### 你要交付的作品

**质量-速度 Pareto 实验板**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：单变量实验员  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 1. 什么叫单变量实验？

如果同时改变 chunk_size、top_k 和 tokenizer，结果发生变化时无法归因。单变量实验固定其他条件，只修改一个变量，并把固定条件一起记录。

本课使用小型本地语料，结论只适用于当前数据版本；方法可以复用到真实大规模语料。

In [1]:
# 导入 Path，用它表示跨平台的文件路径。
from pathlib import Path

# 导入 json，用它读取和保存项目的结构化数据。
import json

# 导入 sys，用它把项目根目录加入 Python 的模块搜索路径。
import sys


# 定义一个函数，负责从当前工作目录向上查找项目根目录。
def find_project_root() -> Path:
    # 把当前目录和它的所有父目录放进候选列表。
    candidates = [Path.cwd(), *Path.cwd().parents]

    # 逐个检查候选目录是否包含本项目的两个核心模块目录。
    for candidate in candidates:
        # 找到同时存在的目录时，返回这个候选目录。
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate

    # 如果所有候选目录都不符合，说明 Jupyter 启动位置不在项目内。
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


# 执行查找函数，得到当前项目根目录。
ROOT = find_project_root()

# 如果项目根目录还不在模块搜索路径中，就把它添加进去。
if str(ROOT) not in sys.path:
    # 把项目根目录插入最前面，确保导入的是当前项目代码。
    sys.path.insert(0, str(ROOT))

# 打印根目录，帮助学习者确认 Notebook 没有在错误目录运行。
print("项目根目录:", ROOT)

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase3.2'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase3.2
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


In [3]:
# 导入计时器，测量搜索调用耗时。
from time import perf_counter

# 导入 KnowledgeBase，构建可复用的生产检索服务。
from phase4_mini_rag_system.knowledge_base import KnowledgeBase

# 导入 Recall 指标，用于判断相关证据是否进入 top-k。
from phase2_semantic_search.metrics import recall_at_k

# 定义固定的原始输入目录。
input_directory = ROOT / "phase1_doc_parser" / "examples" / "input"

# 建立一个固定 Query，后续只改变 top_k 或分块参数。
fixed_query = "Chunk overlap"

# 找出正文中包含 overlap 的 Chunk ID，作为教学相关集合。
reference_kb = KnowledgeBase()
reference_kb.ingest(input_directory, chunk_size=128, overlap=32)
relevant_ids = {str(chunk["id"]) for chunk in reference_kb.chunks if "overlap" in str(chunk["text"]).lower()}

# 确认教学相关集合不为空。
assert relevant_ids

In [4]:
# 定义一个实验函数，固定 Query 和索引，只改变 top_k。
def measure_top_k(top_k):
    # 预热一次当前 Query。
    reference_kb.search(fixed_query, top_k=top_k)

    # 记录计时起点。
    start_time = perf_counter()

    # 执行正式搜索。
    results = reference_kb.search(fixed_query, top_k=top_k)

    # 计算单次搜索耗时。
    elapsed_ms = (perf_counter() - start_time) * 1000

    # 提取排名 ID，供 Recall 计算。
    ranked_ids = [result["chunk_id"] for result in results]

    # 返回这组配置的完整实验记录。
    return {"top_k": top_k, "elapsed_ms": elapsed_ms, "recall": recall_at_k(ranked_ids, relevant_ids, k=top_k), "result_count": len(results)}

# 只改变 top_k，保存三组结果。
top_k_results = [measure_top_k(top_k) for top_k in (1, 2, 5)]

# 打印结果，观察返回数量、质量和耗时的关系。
for row in top_k_results:
    # 输出每组 top_k 的实验记录。
    print(row)

{'top_k': 1, 'elapsed_ms': 0.03789999755099416, 'recall': 1.0, 'result_count': 1}
{'top_k': 2, 'elapsed_ms': 0.029000002541579306, 'recall': 1.0, 'result_count': 2}
{'top_k': 5, 'elapsed_ms': 0.02260001201648265, 'recall': 1.0, 'result_count': 2}


### 读结果时不要预设结论

top_k 增大通常给 Recall 更多机会，但也会带来更多上下文和排序/序列化成本。当前小数据可能看不出明显延迟差异，这不代表大数据上没有成本，只代表本次实验的规模太小。

In [5]:
# 定义要比较的分块配置，其他 Query 和 qrels 保持不变。
chunk_configurations = [(64, 16), (128, 32), (256, 64)]

# 创建空列表，保存分块参数实验结果。
chunk_results = []

# 逐组构建独立知识库，避免索引状态互相污染。
for chunk_size, overlap in chunk_configurations:
    # 创建当前配置的知识库。
    current_kb = KnowledgeBase()

    # 用当前参数导入文档并建立 BM25 索引。
    current_kb.ingest(input_directory, chunk_size=chunk_size, overlap=overlap)

    # 记录计时起点。
    start_time = perf_counter()

    # 执行固定 Query。
    current_results = current_kb.search(fixed_query, top_k=5)

    # 计算当前配置的搜索耗时。
    elapsed_ms = (perf_counter() - start_time) * 1000

    # 提取当前排名 ID。
    current_ids = [result["chunk_id"] for result in current_results]

    # 记录索引规模、质量和单次耗时。
    chunk_results.append({"chunk_size": chunk_size, "overlap": overlap, "chunks": len(current_kb.chunks), "elapsed_ms": elapsed_ms, "recall": recall_at_k(current_ids, relevant_ids, k=5)})

# 打印分块配置对照结果。
for row in chunk_results:
    # 输出一组配置的完整结果。
    print(row)

{'chunk_size': 64, 'overlap': 16, 'chunks': 6, 'elapsed_ms': 0.08339999476447701, 'recall': 0.0}
{'chunk_size': 128, 'overlap': 32, 'chunks': 4, 'elapsed_ms': 0.08450000314041972, 'recall': 1.0}
{'chunk_size': 256, 'overlap': 64, 'chunks': 2, 'elapsed_ms': 0.0641999940853566, 'recall': 0.0}


## 2. 错误归因：指标变差以后先查哪一层？

评估数字只是症状。相关 Chunk 不在 top-k，优先查解析、分块、tokenizer 或召回；相关 Chunk 已经在 top-k 但答案错，才查上下文编排和生成。下一次实验只改对应层，才能建立因果关系。

In [6]:
# 创建一份可复用的错误归因表。
failure_taxonomy = [
    {"observed": "相关 Chunk 不在 top-k", "layer": "parsing/chunking/retrieval", "next_change": "只改分块或召回策略"},
    {"observed": "相关 Chunk 在 top-k，但答案漏掉事实", "layer": "context/generation", "next_change": "只改上下文编排或 prompt"},
    {"observed": "答案出现证据中没有的事实", "layer": "faithfulness", "next_change": "增加证据约束和人工复核"},
]

# 逐条打印归因规则，形成调试习惯。
for failure in failure_taxonomy:
    # 输出观察、归因层和下一步实验变量。
    print(failure)

# 三种错误都必须写明下一步动作。
assert all(failure["next_change"] for failure in failure_taxonomy)

{'observed': '相关 Chunk 不在 top-k', 'layer': 'parsing/chunking/retrieval', 'next_change': '只改分块或召回策略'}
{'observed': '相关 Chunk 在 top-k，但答案漏掉事实', 'layer': 'context/generation', 'next_change': '只改上下文编排或 prompt'}
{'observed': '答案出现证据中没有的事实', 'layer': 'faithfulness', 'next_change': '增加证据约束和人工复核'}


In [7]:
# 组合两个单变量实验的完整记录。
experiment_record = {"fixed_query": fixed_query, "relevant_ids": sorted(relevant_ids), "top_k_results": top_k_results, "chunk_results": chunk_results, "failure_taxonomy": failure_taxonomy}

# 指定实验记录路径。
experiment_path = ROOT / "data" / "processed" / "phase3_single_variable_experiments.json"

# 保存实验条件和结果，供下一课生成报告。
experiment_path.write_text(json.dumps(experiment_record, ensure_ascii=False, indent=2), encoding="utf-8")

# 打印交付路径。
print("已生成:", experiment_path)

已生成: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\data\processed\phase3_single_variable_experiments.json


## 本课验收

- [ ] 能指出每个实验固定了什么、改变了什么。
- [ ] 能同时查看 Recall 和延迟，而不是只追求一个数字。
- [ ] 能把一次失败映射到下一步实验。
- [ ] 已生成 `phase3_single_variable_experiments.json`。

## Boss Challenge：只改 top_k，画出质量与延迟变化，并写一句‘在什么范围内结论成立’。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [8]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [9]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['data/processed/phase3_single_variable_experiments.json']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: ['data\\processed\\phase3_single_variable_experiments.json']
